# HamsAI RAG Fine-tuning
Run on Google Colab with T4/L4 GPU. Upload `train_split.json`, `val_split.json`, `retrieval_pairs.json` when prompted.

In [ ]:
# Cell 1 — Install dependencies (run once, then Runtime → Restart session)
import subprocess, sys

pkgs = [
    "sentence-transformers==3.3.1",
    "transformers==4.46.3",
    "huggingface_hub==0.26.2",
    "accelerate==1.1.1",
    "FlagEmbedding==1.3.3",
    "peft==0.13.2",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("\n✓ All packages installed. Now go to Runtime → Restart session, then run from Cell 2.")

In [ ]:
# Cell 2 — Upload data files
from google.colab import files
import json

print("Upload these 3 files: train_split.json, val_split.json, retrieval_pairs.json")
uploaded = files.upload()

train = json.load(open("train_split.json", encoding="utf-8"))
val   = json.load(open("val_split.json",   encoding="utf-8"))
test  = json.load(open("retrieval_pairs.json", encoding="utf-8"))

print(f"\n✓ Loaded — train: {len(train)}, val: {len(val)}, test: {len(test)}")

In [ ]:
# Cell 3 — Evaluate BASE embedding model (before fine-tuning)
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import torch, gc

BASE_EMBEDDING = "BAAI/bge-m3"
EMBEDDING_OUT  = "./bge-m3-hamsai-finetuned"

def evaluate_triplet(model, rows, label):
    correct = total = 0
    for item in tqdm(rows, desc=label):
        q = item["query"]
        p = item["positive"]
        n = item["hard_negative"]
        emb = model.encode([q, p, n], normalize_embeddings=True)
        if float(emb[0] @ emb[1]) > float(emb[0] @ emb[2]):
            correct += 1
        total += 1
    acc = correct / total if total else 0.0
    print(f"  {label}: {correct}/{total} = {acc:.4f}")
    return acc

print("Loading base embedding model...")
base_embedding = SentenceTransformer(BASE_EMBEDDING)
base_embedding.max_seq_length = 512

before_embedding_acc = evaluate_triplet(base_embedding, test, "embedding_before")

del base_embedding
gc.collect()
torch.cuda.empty_cache()
print("\n✓ Baseline recorded. Proceed to Cell 4.")

In [ ]:
# Cell 4 — Fine-tune embedding model with TripletLoss
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import TripletLoss
from datasets import Dataset
import torch, gc

print("Loading model for fine-tuning...")
embedding_model = SentenceTransformer(BASE_EMBEDDING)
embedding_model.max_seq_length = 512

# Build dataset
train_dataset = Dataset.from_dict({
    "anchor"  : [r["query"]         for r in train],
    "positive": [r["positive"]      for r in train],
    "negative": [r["hard_negative"] for r in train],
})
val_dataset = Dataset.from_dict({
    "anchor"  : [r["query"]         for r in val],
    "positive": [r["positive"]      for r in val],
    "negative": [r["hard_negative"] for r in val],
})

loss = TripletLoss(model=embedding_model)

args = SentenceTransformerTrainingArguments(
    output_dir=EMBEDDING_OUT,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=loss,
)

print("Fine-tuning embedding model (3 epochs, ~45 min on T4)...")
trainer.train()
embedding_model.save_pretrained(EMBEDDING_OUT)

# Evaluate after
finetuned_embedding = SentenceTransformer(EMBEDDING_OUT)
after_embedding_acc = evaluate_triplet(finetuned_embedding, test, "embedding_after")

print(f"\n✓ Embedding improvement: {before_embedding_acc:.4f} → {after_embedding_acc:.4f} (+{after_embedding_acc - before_embedding_acc:.4f})")

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 5 — Fine-tune reranker with BCE loss
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CERerankingEvaluator
from sentence_transformers import InputExample
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torch, gc

BASE_RERANKER  = "BAAI/bge-reranker-v2-m3"
RERANKER_OUT   = "./bge-reranker-hamsai-finetuned"

def evaluate_reranker(model, rows, label):
    correct = total = 0
    for r in tqdm(rows, desc=label):
        scores = model.predict([[r["query"], r["positive"]],
                                [r["query"], r["hard_negative"]]])
        correct += int(scores[0] > scores[1])
        total   += 1
    acc = correct / total if total else 0.0
    print(f"  {label}: {correct}/{total} = {acc:.4f}")
    return acc

print("Loading base reranker...")
base_reranker = CrossEncoder(BASE_RERANKER, num_labels=1, max_length=512)
before_reranker_acc = evaluate_reranker(base_reranker, test, "reranker_before")
del base_reranker
gc.collect()
torch.cuda.empty_cache()

print("\nFine-tuning reranker (2 epochs, ~35 min on T4)...")
reranker = CrossEncoder(BASE_RERANKER, num_labels=1, max_length=512)

reranker_examples = []
for r in train:
    reranker_examples.append(InputExample(texts=[r["query"], r["positive"]],      label=1.0))
    reranker_examples.append(InputExample(texts=[r["query"], r["hard_negative"]], label=0.0))

reranker_loader = DataLoader(reranker_examples, shuffle=True, batch_size=4)

samples = {
    str(i): {"query": r["query"], "positive": [r["positive"]], "negative": [r["hard_negative"]]}
    for i, r in enumerate(val)
}
reranker_eval = CERerankingEvaluator(samples=samples, name="hamsai_reranker_val")

reranker.fit(
    train_dataloader=reranker_loader,
    evaluator=reranker_eval,
    epochs=2,
    warmup_steps=max(10, len(reranker_loader) // 10),
    optimizer_params={"lr": 1e-5},
    output_path=RERANKER_OUT,
    show_progress_bar=True,
)

finetuned_reranker = CrossEncoder(RERANKER_OUT, num_labels=1, max_length=512)
after_reranker_acc = evaluate_reranker(finetuned_reranker, test, "reranker_after")

print(f"\n✓ Reranker improvement: {before_reranker_acc:.4f} → {after_reranker_acc:.4f} (+{after_reranker_acc - before_reranker_acc:.4f})")

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 6 — Push to HuggingFace and download results
from huggingface_hub import notebook_login
from google.colab import files
import json

notebook_login()

HF_USERNAME = "ani122312"  # ← change if your HF username is different

finetuned_embedding.push_to_hub(f"{HF_USERNAME}/bge-m3-hamsai-finetuned", private=True)
finetuned_reranker.push_to_hub(f"{HF_USERNAME}/bge-reranker-hamsai-finetuned", private=True)

results = {
    "embedding": {
        "base_model"       : BASE_EMBEDDING,
        "before_accuracy"  : before_embedding_acc,
        "after_accuracy"   : after_embedding_acc,
        "improvement"      : round(after_embedding_acc - before_embedding_acc, 4),
    },
    "reranker": {
        "base_model"       : BASE_RERANKER,
        "before_accuracy"  : before_reranker_acc,
        "after_accuracy"   : after_reranker_acc,
        "improvement"      : round(after_reranker_acc - before_reranker_acc, 4),
    },
}
json.dump(results, open("finetuning_results.json", "w"), indent=2)
files.download("finetuning_results.json")

print(json.dumps(results, indent=2))
print("\n✓ Models pushed to HuggingFace. Results downloaded.")